# Recession Impact Analysis
## Data Collection & Cleaning

Downloads official U.S. employment data from FRED, cleans it, and saves one master table.

---


### Step 1: Load tools

Python libraries for data handling, date alignment, and FRED API connection.


In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from fredapi import Fred

### Step 2: Connect to FRED

You need a free FRED API key. If you do not have one, sign up here:
https://fred.stlouisfed.org/docs/api/api_key.html

Paste your key into the `.env` file in the project folder as:
`FRED_API_KEY=your_actual_key_here`


In [3]:
env_path = ".env" if os.path.exists(".env") else os.path.join("..", ".env")
load_dotenv(env_path)
FRED_API_KEY = os.getenv("FRED_API_KEY")

if not FRED_API_KEY or FRED_API_KEY == "your_fred_api_key_here":
    raise ValueError("Please set your FRED_API_KEY in the .env file before running this notebook")

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection ready")

FRED connection ready


### Step 3: Download all 6 FRED series fresh

Raw series fetched directly from FRED with no cached CSV merge.


In [4]:
tickers = {
    "USREC": "recession_indicator",
    "PAYEMS": "total_nonfarm_payroll",
    "USCONS": "construction_employment",
    "MANEMP": "manufacturing_employment",
    "USPBS": "professional_business_services",
    "USINFO": "information_sector",
}

raw_series = {}
for fred_id, col_name in tickers.items():
    print(f"Downloading {fred_id}...")
    s = fred.get_series(fred_id)
    raw_series[col_name] = s
    print(f"   {len(s)} observations from {s.index.min().date()} to {s.index.max().date()}")

print("\nAll 6 series downloaded")

   2063 observations from 1854-12-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01
   1054 observations from 1939-01-01 to 2026-10-01

All 6 series downloaded


### Step 4: Find overlapping start date

Determine the first date where ALL 6 series have data. This avoids pre-1939 rows
where only USREC exists and employment columns are empty.


In [5]:
first_dates = {}
for col_name, series in raw_series.items():
    first_dates[col_name] = series.index.min()
    print(f"{col_name}: starts {series.index.min().date()}")

overlapping_start_date = max(first_dates.values())
print(f"\nActual overlapping start date: {overlapping_start_date.date()}")
print(f"All 6 tickers have data from this date forward.")

recession_indicator: starts 1854-12-01
total_nonfarm_payroll: starts 1939-01-01
construction_employment: starts 1939-01-01
manufacturing_employment: starts 1939-01-01
professional_business_services: starts 1939-01-01
information_sector: starts 1939-01-01

Actual overlapping start date: 1939-01-01
All 6 tickers have data from this date forward.


### Step 5: Clean and align to monthly frequency

Resample each series to month-end, forward-fill short gaps, then slice to the
overlapping start date so all 6 columns have complete overlapping data.


In [6]:
cleaned = {}
for col_name, series in raw_series.items():
    monthly = series.resample("ME").last()
    monthly = monthly.ffill()
    cleaned[col_name] = monthly.loc[overlapping_start_date:]

print(f"All series cleaned to monthly frequency from {overlapping_start_date.date()}")
print(f"Rows per series: {len(cleaned[list(cleaned.keys())[0]])}")

All series cleaned to monthly frequency from 1939-01-01
Rows per series: 1054


### Step 6: Merge into master table

Stack all 6 raw series side by side into one table indexed by date.


In [7]:
master = pd.concat(cleaned, axis=1)
master.index.name = "date"
master = master.dropna(how="any")

print(f"Master table shape: {master.shape}")
print(f"Date range: {master.index.min().date()} to {master.index.max().date()}")
print(f"Columns: {list(master.columns)}")
print(f"Any NaN remaining: {master.isna().any().any()}")

Master table shape: (1054, 6)
Date range: 1939-01-31 to 2026-10-31
Columns: ['recession_indicator', 'total_nonfarm_payroll', 'construction_employment', 'manufacturing_employment', 'professional_business_services', 'information_sector']
Any NaN remaining: False


### Step 7: Feature engineering

Create blue-collar and white-collar job aggregates plus their percentage shares
of the core cohort.


In [8]:
master["blue_collar_jobs"] = master["construction_employment"] + master["manufacturing_employment"]
master["white_collar_jobs"] = master["professional_business_services"] + master["information_sector"]
master["core_cohort_total"] = master["blue_collar_jobs"] + master["white_collar_jobs"]
master["white_collar_pct"] = (master["white_collar_jobs"] / master["core_cohort_total"]) * 100
master["blue_collar_pct"] = (master["blue_collar_jobs"] / master["core_cohort_total"]) * 100

print("Feature engineering complete")
print("New columns: blue_collar_jobs, white_collar_jobs, core_cohort_total, white_collar_pct, blue_collar_pct")
print(master[['blue_collar_jobs', 'white_collar_jobs', 'white_collar_pct', 'blue_collar_pct']].head())

Feature engineering complete
New columns: blue_collar_jobs, white_collar_jobs, core_cohort_total, white_collar_pct, blue_collar_pct
            blue_collar_jobs  white_collar_jobs  white_collar_pct  \
date                                                                
1939-01-31           10216.0             3055.0         23.020119   
1939-02-28           10337.0             3068.0         22.886982   
1939-03-31           10500.0             3084.0         22.703180   
1939-04-30           10558.0             3084.0         22.606656   
1939-05-31           10561.0             3093.0         22.652703   

            blue_collar_pct  
date                         
1939-01-31        76.979881  
1939-02-28        77.113018  
1939-03-31        77.296820  
1939-04-30        77.393344  
1939-05-31        77.347297  


### Step 8: Save master table

Saves the clean dataframe to `data/processed/master_table.csv` with automatic
folder creation if the directory does not exist.


In [9]:
# Determine project base path safely whether run from notebooks/ or root
base_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
out_path = os.path.join(base_dir, "data", "processed", "master_table.csv")
abs_path = os.path.abspath(out_path)
print(f"Attempting to save to: {abs_path}")
os.makedirs(os.path.dirname(abs_path), exist_ok=True)
print(f"Directory created/exists: {os.path.dirname(abs_path)}")
master.to_csv(abs_path)

print(f"\nSaved master table to {abs_path}")
print(f"Shape: {master.shape[0]} rows x {master.shape[1]} columns")
print(f"Date range: {master.index.min().date()} to {master.index.max().date()}")
print(master.head(10))

Attempting to save to: c:\Users\vansh\OneDrive\Desktop\recessionImpact\data\processed\master_table.csv
Directory created/exists: c:\Users\vansh\OneDrive\Desktop\recessionImpact\data\processed

Saved master table to c:\Users\vansh\OneDrive\Desktop\recessionImpact\data\processed\master_table.csv
Shape: 1054 rows x 11 columns
Date range: 1939-01-31 to 2026-10-31
            recession_indicator  total_nonfarm_payroll  \
date                                                     
1939-01-31                  0.0                29923.0   
1939-02-28                  0.0                30100.0   
1939-03-31                  0.0                30280.0   
1939-04-30                  0.0                30094.0   
1939-05-31                  0.0                30299.0   
1939-06-30                  0.0                30502.0   
1939-07-31                  0.0                30419.0   
1939-08-31                  0.0                30663.0   
1939-09-30                  0.0                31031.0   


### Step 9: Final verification

Confirms the saved file is readable and aligned for Notebooks 2 and 3.


In [10]:
verify = pd.read_csv(abs_path, parse_dates=["date"], index_col="date")
print("Verification of saved CSV:")
verify.info()
print(f"\nFirst 5 rows:")
print(verify.head())
print(f"\nLast 5 rows:")
print(verify.tail())
print(f"\nColumn check:")
expected_cols = [
    "recession_indicator", "total_nonfarm_payroll", "construction_employment",
    "manufacturing_employment", "professional_business_services", "information_sector",
    "blue_collar_jobs", "white_collar_jobs", "core_cohort_total",
    "white_collar_pct", "blue_collar_pct"
]
print(f"Expected columns: {len(expected_cols)}")
print(f"Actual columns: {len(verify.columns)}")
print(f"All present: {set(expected_cols).issubset(set(verify.columns))}")

Verification of saved CSV:
<class 'pandas.DataFrame'>
DatetimeIndex: 1054 entries, 1939-01-31 to 2026-10-31
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   recession_indicator             1054 non-null   float64
 1   total_nonfarm_payroll           1054 non-null   float64
 2   construction_employment         1054 non-null   float64
 3   manufacturing_employment        1054 non-null   float64
 4   professional_business_services  1054 non-null   float64
 5   information_sector              1054 non-null   float64
 6   blue_collar_jobs                1054 non-null   float64
 7   white_collar_jobs               1054 non-null   float64
 8   core_cohort_total               1054 non-null   float64
 9   white_collar_pct                1054 non-null   float64
 10  blue_collar_pct                 1054 non-null   float64
dtypes: float64(11)
memory usage: 98.8 KB

First 5 rows:
         